# Talk 7   

Aquí tienes la traducción al español del resumen:

---



## PyCon Colombia 2026

**Inteligencia Artificial | Aprendizaje Automático | Ciencia de Datos | Core Python**

### Tu evaluación de IA te está mintiendo (*Your AI Eval Is Lying To You*)

* **FORMATO:** Charla
* **NIVEL:** Intermedio
* **IDIOMA:** Inglés

Cuando defines `temperature=0` y ejecutas tu evaluación de IA, esperas que la misma entrada genere la misma salida. No es así. Mediciones recientes en Qwen3-235B a `temperature=0` produjeron 80 respuestas únicas para un solo *prompt*. Entonces, cuando tu evaluación reporta una "tasa de aprobación del 92%", ¿qué significa realmente? ¿Es un 92% capaz, un 92% confiable o un 92% afortunada en una muestra pequeña?

Esta charla aborda la brecha entre la forma en que el ecosistema de evaluación de IA habla de los puntajes y lo que esos puntajes realmente pueden respaldar. Recorreremos cinco herramientas específicas que corrigen esta brecha, todas basadas en métodos publicados:

1. **Pass@k frente a pass^k:** capacidad versus confiabilidad, dos cuestiones distintas que un solo número oculta (Chen et al. 2021, artículo de OpenAI Codex).
2. **Intervalos de confianza de Wilson con manejo adecuado de límites:** para que tu "92%" venga acompañado de un rango honesto (Brown et al. 2001).
3. **Pass@k Bayesiano con conjugación Beta-Binomial:** para cuando deseas una distribución *a posteriori* en lugar de una estimación puntual (Hariri et al., ICLR 2026).
4. **Detección secuencial de desviación (*drift*) con EWMA, CUSUM y OLS:** para detectar la regresión en las evaluaciones mientras sea pequeña, en lugar de hacerlo después de que un cliente la reporte (Lucas-Saccucci 1990, Page 1954, Montgomery 2012).
5. **Control de tasa de falsos descubrimientos (FDR) mediante Benjamini-Hochberg, Benjamini-Yekutieli y e-BH:** para cuando ejecutas múltiples verificaciones de desviación correlacionadas en paralelo y no quieres falsas alarmas (Benjamini-Hochberg 1995, Wang-Ramdas 2022).

Cada método incluye una breve demostración en Python puro, sin dependencias de *frameworks*. La audiencia se llevará implementaciones de referencia que podrá pegar esta misma noche en una configuración existente de `pytest`.

La charla también ofrece un adelanto de un *plugin* de `pytest` de código abierto (*open-core*) que se lanzará en julio de 2026, el cual empaqueta estos métodos en un único flujo de evaluación con reportes SARIF y un flujo de trabajo para regresiones de línea base. La presentación muestra las primitivas abiertas y la metodología que las impulsa.

El ecosistema de evaluación de IA en producción (LangSmith, Arize Phoenix, Evidently, DeepEval, Promptfoo, entre otros) utiliza mayoritariamente umbrales absolutos y promedios simples. Ninguna de las diez plataformas que analicé combina pruebas secuenciales con control FDR en escalas de puntuación acotadas. El enfoque de esta charla no es competitivo; se trata de una brecha metodológica con la que cualquier equipo que despliegue evaluaciones de IA en producción se cruzará tarde o temprano.

---

### Ponente

**Sankalp Gilda**

*Staff MLE @ DeepThought Solutions*

Sankalp Gilda, PhD (Astrofísica, Universidad de Florida, 2021). *Staff Machine Learning Engineer* en DeepThought Solutions, donde lidera el trabajo en herramientas de evaluación de IA en producción, instrumentación del lado del host para entornos aislados (*sandboxes*) de ejecución agéntica, y extracción de grafos de conocimiento basados en LLM.

# Explicación del tema de la charla  

Esta charla aborda un problema muy común en la inteligencia artificial actual: **los resultados de los modelos de lenguaje (LLMs) son inconsistentes**, pero nuestras métricas de evaluación suelen ser demasiado simplistas para notarlo.



Aquí te explico la idea principal en sencillo y cómo se traducen esos conceptos matemáticos a código Python básico.

---



## El problema en palabras simples

Cuando le pides algo a un modelo y pones la `temperatura = 0`, la teoría dice que debería darte **siempre la misma respuesta exacta**. Sin embargo, por cómo funcionan las tarjetas gráficas (procesamiento en paralelo, redondeos de números flotantes), el modelo produce pequeñas variaciones.

Si evalúas tu modelo con 10 preguntas y responde bien 9, dices: *"¡Genial, 90% de éxito!"*. Pero esta charla demuestra que ese 90% puede ser un espejismo:

* ¿El modelo es verdaderamente inteligente (**capaz**) o solo tuvo suerte en ese intento?
* Si lo ejecutas 100 veces seguidas, ¿mantiene la calidad (**confiable**)?

---



## Los conceptos clave explicados con Python

A continuación verás ejemplos minimalistas en Python puro (sin librerías raras) de las técnicas que propone el ponente:

### 1. `Pass@k` vs `Pass^k`: Capacidad vs. Confiabilidad

* **`Pass@k` (Capacidad):** ¿El modelo es capaz de dar la respuesta correcta *al menos una vez* en $k$ intentos?
* **`Pass^k` (Confiabilidad):** ¿El modelo da la respuesta correcta *en todos* los $k$ intentos seguidos?




In [1]:
# Imaginemos que ejecutamos el modelo 5 veces para la misma pregunta:
# 1 = Respuesta correcta, 0 = Respuesta incorrecta
intentos = [1, 0, 1, 1, 0]

# Pass@k: ¿Logró responder bien al menos una vez?
capaz = 1 in intentos  # True (Sí es capaz)

# Pass^k: ¿Respondió bien en TODOS los intentos?
confiable = all(res == 1 for res in intentos)  # False (No es totalmente confiable)

print(f"¿Es capaz?: {capaz}")
print(f"¿Es confiable?: {confiable}")


¿Es capaz?: True
¿Es confiable?: False



> **Conclusión:** Un modelo puede tener un `Pass@k` alto (sabe la respuesta) pero un `Pass^k` bajo (no es estable para producción).

---



### 2. Intervalo de Confianza: Un "92%" honesto

Si pruebas tu IA solo 10 veces y acierta 9, decir "tengo 90% de precisión" es engañoso porque la muestra es muy pequeña. Necesitas calcular un **rango de incertidumbre** (el intervalo de Wilson).



In [2]:
import math

def intervalo_confianza_simple(éxitos, total):
    """Calcula un rango de certeza estimado para un porcentaje."""
    p_estimada = éxitos / total
    # Error estándar básico
    error = 1.96 * math.sqrt((p_estimada * (1 - p_estimada)) / total)
    
    limite_inferior = max(0.0, p_estimada - error)
    limite_superior = min(1.0, p_estimada + error)
    
    return limite_inferior, limite_superior

# Ejemplo con muestra pequeña (9 acertadas de 10)
inf, sup = intervalo_confianza_simple(éxitos=9, total=10)
print(f"Resultado de 9/10: 90% real está entre {inf:.1%} y {sup:.1%}")
# ¡Verás que el rango es enorme! Puede ir desde ~71% hasta ~100%

# Ejemplo con muestra grande (900 acertadas de 1000)
inf_g, sup_g = intervalo_confianza_simple(éxitos=900, total=1000)
print(f"Resultado de 900/1000: 90% real está entre {inf_g:.1%} y {sup_g:.1%}")
# Aquí el rango es mucho más estrecho y confiable (~88% a ~92%)


Resultado de 9/10: 90% real está entre 71.4% y 100.0%
Resultado de 900/1000: 90% real está entre 88.1% y 91.9%



# 3. Detección de Desviaciones (CUSUM)

En producción, un modelo de IA puede empeorar sin que te des cuenta (por ejemplo, tras una actualización). 

En lugar de esperar a que un cliente se queje, la técnica **CUSUM** suma los errores de forma acumulativa para detectar caídas de rendimiento sutiles pero constantes.


In [3]:
def detectar_caida_rendimiento(puntuaciones, meta=0.90):
    """Detecta si el rendimiento promedio está cayendo acumulativamente."""
    suma_acumulada = 0
    umbral_alerta = 1.5  # Límite para disparar la alarma
    
    for i, puntaje in enumerate(puntuaciones):
        # Medimos cuánto se desvía de la meta esperada
        desviacion = meta - puntaje
        suma_acumulada = max(0, suma_acumulada + desviacion)
        
        if suma_acumulated := suma_acumulada > umbral_alerta:
            print(f"⚠️ ¡ALERTA! Degradación detectada en la prueba #{i + 1}")
            return True
            
    print("✅ El sistema se mantiene estable.")
    return False

# Historial donde el modelo empieza a fallar gradualmente:
historial = [0.9, 0.9, 0.85, 0.7, 0.75, 0.6, 0.5]
detectar_caida_rendimiento(historial)


✅ El sistema se mantiene estable.


False


# ¿Qué se lleva la audiencia?

El ponente busca que los desarrolladores dejen de confiar en métricas "ciegas" y aprendan a crear **pruebas unitarias automáticas con `pytest**` que les digan la verdad sobre su IA antes de desplegarla a clientes reales.